# RAG-Anything on Google Colab

Multimodal RAG pipeline — processes documents containing text, tables, charts, and formulas.

**Before running:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Add your OpenAI API key to Colab Secrets (🔑 icon in left sidebar) with the name `OPENAI_API_KEY`
3. Upload your documents to Google Drive under `My Drive/ragAnything/documents/`

**Drive folder structure created automatically:**
```
My Drive/
└── ragAnything/
    ├── documents/     ← put your PDFs here
    ├── rag_storage/   ← knowledge graph (persists across sessions)
    └── output/        ← MinerU parse output
```

## Step 1 — Install dependencies

In [1]:
!pip install -q 'raganything[text]' python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.6/786.6 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 2.5 MB/s eta 0

## Step 2 — Mount Google Drive

In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/ragAnything'
DOCS_DIR = f'{BASE_DIR}/documents'
RAG_STORAGE = f'{BASE_DIR}/rag_storage'
OUTPUT_DIR = f'{BASE_DIR}/output'

os.makedirs(DOCS_DIR, exist_ok=True)
os.makedirs(RAG_STORAGE, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Drive mounted.')
print(f'Documents folder: {DOCS_DIR}')
docs = [f for f in os.listdir(DOCS_DIR) if not f.startswith('.')]
print(f'Documents found: {docs if docs else "none yet — upload PDFs to the documents folder"}')

Mounted at /content/drive
Drive mounted.
Documents folder: /content/drive/MyDrive/ragAnything/documents
Documents found: ['sklanguser.pdf', 'sklangref.pdf']


## Step 3 — Configuration

API key is loaded from Colab Secrets. Add it via the 🔑 icon in the left sidebar with name `OPENAI_API_KEY`.

In [3]:
import os
from google.colab import userdata

# Suppress TensorFlow oneDNN info message — it goes to stderr and causes
# raganything to falsely treat a successful MinerU run as a failure.
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

try:
    API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    import getpass
    API_KEY = getpass.getpass('Enter your OpenAI API key: ')

LLM_MODEL       = 'gpt-5.4-nano'
VISION_MODEL    = 'gpt-5.4-nano'
EMBEDDING_MODEL = 'text-embedding-3-large'
EMBEDDING_DIM   = 3072

MINERU_BACKEND  = 'pipeline'
MINERU_FORMULA  = True
PARSE_METHOD    = 'auto'

# ── Test mode ──────────────────────────────────────────────────────────────
TEST_MODE  = False   # False = full documents
TEST_PAGES = 10
# ───────────────────────────────────────────────────────────────────────────

print('Configuration ready.')
print(f'LLM: {LLM_MODEL}  |  Vision: {VISION_MODEL}  |  Embedding: {EMBEDDING_MODEL}')
print(f'Backend: {MINERU_BACKEND}  |  Formula: {MINERU_FORMULA}  |  Parse method: {PARSE_METHOD}')
print(f'Test mode: {TEST_MODE}' + (f'  →  first {TEST_PAGES} pages per PDF' if TEST_MODE else '  →  full documents'))

Configuration ready.
LLM: gpt-5.4-nano  |  Vision: gpt-5.4-nano  |  Embedding: text-embedding-3-large
Backend: pipeline  |  Formula: True  |  Parse method: auto
Test mode: False  →  full documents


## Step 4 — Initialise RAG

In [ ]:
import os
import torch
from functools import partial
from pathlib import Path

from raganything import RAGAnything, RAGAnythingConfig
from lightrag.llm.openai import openai_complete_if_cache, openai_embed
from lightrag.utils import EmbeddingFunc

# Confirm GPU is available for MinerU
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f'GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    DEVICE = 'cpu'
    print('WARNING: No GPU detected — MinerU will run on CPU (slow). Check Runtime → Change runtime type.')


def llm_model_func(prompt, system_prompt=None, history_messages=[], **kwargs):
    return openai_complete_if_cache(
        LLM_MODEL, prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=API_KEY,
        **kwargs,
    )


def vision_model_func(
    prompt, system_prompt=None, history_messages=[], image_data=None, messages=None, **kwargs
):
    if messages:
        return openai_complete_if_cache(
            VISION_MODEL, '', messages=messages, api_key=API_KEY, **kwargs
        )
    if image_data:
        return openai_complete_if_cache(
            VISION_MODEL, '',
            messages=[
                {'role': 'system', 'content': system_prompt} if system_prompt else None,
                {'role': 'user', 'content': [
                    {'type': 'text', 'text': prompt},
                    {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{image_data}'}},
                ]},
            ],
            api_key=API_KEY, **kwargs,
        )
    return llm_model_func(prompt, system_prompt, history_messages, **kwargs)


embedding_func = EmbeddingFunc(
    embedding_dim=EMBEDDING_DIM,
    max_token_size=8192,
    func=partial(openai_embed.func, model=EMBEDDING_MODEL, api_key=API_KEY),
)

config = RAGAnythingConfig(
    working_dir=RAG_STORAGE,
    parser_output_dir=OUTPUT_DIR,
    parse_method=PARSE_METHOD,
    enable_image_processing=True,
    enable_table_processing=True,
    enable_equation_processing=MINERU_FORMULA,
    max_concurrent_files=1,  # one at a time — avoids two MinerU instances competing for GPU
)

rag = RAGAnything(
    config=config,
    llm_model_func=llm_model_func,
    vision_model_func=vision_model_func,
    embedding_func=embedding_func,
)

# Load LightRAG from existing storage — allows querying without re-running Step 5
await rag._ensure_lightrag_initialized()

print(f'RAGAnything initialised and ready  |  Parser device: {DEVICE}')

## Step 5 — Process documents

Indexes all supported files in the `documents/` folder into the knowledge graph.  
Already-indexed documents are skipped automatically on re-runs.

In [ ]:
import asyncio
import tempfile
from pathlib import Path
from pypdf import PdfReader, PdfWriter

SUPPORTED_EXTENSIONS = set(config.supported_file_extensions)

def slice_pdf(src: Path, pages: int) -> Path:
    """Return a temporary PDF containing only the first `pages` pages."""
    reader = PdfReader(str(src))
    writer = PdfWriter()
    for i in range(min(pages, len(reader.pages))):
        writer.add_page(reader.pages[i])
    tmp = tempfile.NamedTemporaryFile(suffix='.pdf', delete=False)
    writer.write(tmp)
    tmp.close()
    return Path(tmp.name)

documents = sorted(
    f for f in Path(DOCS_DIR).rglob('*')
    if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS and not f.name.startswith('.')
)

if not documents:
    print(f'No supported documents found in {DOCS_DIR}')
else:
    print(f'Found {len(documents)} document(s):')
    for doc in documents:
        reader = PdfReader(str(doc))
        pages_to_process = min(TEST_PAGES, len(reader.pages)) if TEST_MODE else len(reader.pages)
        print(f'  - {doc.name}  ({len(reader.pages)} pages total → processing {pages_to_process})')

    async def process_all():
        for i, doc in enumerate(documents, 1):
            if TEST_MODE and doc.suffix.lower() == '.pdf':
                target = slice_pdf(doc, TEST_PAGES)
                print(f'\n[{i}/{len(documents)}] Processing: {doc.name} (first {TEST_PAGES} pages)')
            else:
                target = doc
                print(f'\n[{i}/{len(documents)}] Processing: {doc.name}')

            await rag.process_document_complete(
                file_path=str(target),
                backend=MINERU_BACKEND,
                formula=MINERU_FORMULA,
                table=False,       # MinerU's internal table model crashes on GPU; raganything handles tables separately
                device=DEVICE,     # 'cuda' = use the GPU for parsing
            )

            if TEST_MODE and target != doc:
                target.unlink(missing_ok=True)  # clean up temp file

        print('\nAll documents processed.')

    await process_all()

## Step 6 — Query

Change `QUERY` and re-run this cell as many times as you like.

In [ ]:
QUERY = 'What is data manipulation?'     # ← change this
MODE  = 'naive'                          # naive = vector search only, smallest context
                                         # local / global / hybrid = larger context, may hit token limits

result = await rag.aquery(QUERY, mode=MODE, top_k=5)
print(result)

## (Optional) Reset knowledge graph

Run this cell only if you want to clear all indexed data and start fresh.

In [ ]:
import shutil

confirm = input('Type YES to delete all indexed data: ')
if confirm.strip().upper() == 'YES':
    shutil.rmtree(RAG_STORAGE, ignore_errors=True)
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    os.makedirs(RAG_STORAGE, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print('Knowledge graph cleared. Re-run Step 4 to reinitialise.')
else:
    print('Cancelled.')